In [16]:
import pandas as pd

In [3]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [ ]:
df = pd.read_csv("OECD_RD_GDP_2000_2025.csv")
df2 = pd.read_csv("WorldBank_Public_Debt_GDP_2000_2025.csv")
df3 = pd.read_csv("OECD_Productivity_Variation_2000_2025.csv")
df4 = pd.read_csv("OECD_Tertiary_Education_2000_2025.csv")
df5 = pd.read_csv("OECD_Unemployment_Rate_2000_2025.csv")
df6 = pd.read_csv("OECD_Inflation_CPI_2000_2025.csv")
dft = pd.read_csv("OECD_Trust_Institutions_2000_2025.csv")
dfe = pd.read_csv("WorldBank_Energy_Import_Dependency_2000_2025.csv")
dfg = pd.read_csv("OECD_GDP_Growth_2000_2025.csv")



In [72]:
dfe

,Country,Year,Value
0,AL,2000,45.781
1,AL,2001,52.210
2,AL,2002,53.360
3,AL,2003,50.757
4,AL,2004,47.764
...,...,...,...
942,XK,2018,29.274
943,XK,2019,30.534
944,XK,2020,29.537
945,XK,2021,32.614


In [73]:
pip install pycountry

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pycountry

def convert_iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

In [19]:
dfe["Country"] = dfe["Country"].apply(convert_iso2_to_iso3)

In [76]:
dfe

,Country,Year,Value
0,ALB,2000,45.781
1,ALB,2001,52.210
2,ALB,2002,53.360
3,ALB,2003,50.757
4,ALB,2004,47.764
...,...,...,...
942,NaN,2018,29.274
943,NaN,2019,30.534
944,NaN,2020,29.537
945,NaN,2021,32.614


In [20]:
#Check inconsistent text
df["Country"] = df["Country"].str.lower().str.strip()

In [77]:
dfe

,Country,Year,Value
0,ALB,2000,45.781
1,ALB,2001,52.210
2,ALB,2002,53.360
3,ALB,2003,50.757
4,ALB,2004,47.764
...,...,...,...
942,NaN,2018,29.274
943,NaN,2019,30.534
944,NaN,2020,29.537
945,NaN,2021,32.614


In [ ]:
import pandas as pd
import pycountry

files = [
    "OECD_RD_GDP_2000_2025.csv",
    "WorldBank_Public_Debt_GDP_2000_2025.csv",
    "OECD_Productivity_Variation_2000_2025.csv",
    "OECD_Tertiary_Education_2000_2025.csv",
    "OECD_Unemployment_Rate_2000_2025.csv",
    "OECD_Inflation_CPI_2000_2025.csv",
    "OECD_Trust_Institutions_2000_2025.csv",
    "WorldBank_Energy_Import_Dependency_2000_2025.csv",
    "OECD_GDP_Growth_2000_2025.csv"
]


# =========================
# COUNTRY CODE FIX
# =========================
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None


# =========================
# CLEANING FUNCTION
# =========================
def clean_dataset(df, fix_iso2=False):

    df.columns = df.columns.str.lower().str.strip()

    df = df[["country", "year", "value"]]

    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.drop_duplicates()
    df = df.sort_values(["country", "year"])

    # 🔥 FIX ISO2 → ISO3 ONLY IF NEEDED
    if fix_iso2:
        df["country"] = df["country"].apply(iso2_to_iso3)

    df["value"] = df.groupby("country")["value"].transform(
        lambda x: x.interpolate().ffill().bfill()
    )

    return df


# =========================
# LOAD + CLEAN
# =========================
datasets = {}

for file in files:
    df = pd.read_csv(file)

    name = file.replace(".csv", "")

    # 🔥 detect Eurostat dataset
    if "Eurostat" in file:
        datasets[name] = clean_dataset(df, fix_iso2=True)
    else:
        datasets[name] = clean_dataset(df, fix_iso2=False)


# =========================
# PIVOT (SAFE VERSION)
# =========================
pivot_datasets = {}

for name, df in datasets.items():
    pivot_datasets[name] = df.pivot_table(
        index="year",
        columns="country",
        values="value",
        aggfunc="mean"
    ).sort_index()

In [40]:
df2

,Country,Year,Value
0,AUS,2014,38.063
1,AUS,2015,39.844
2,AUS,2016,41.415
3,AUS,2017,43.022
4,AUS,2018,42.983
...,...,...,...
783,BGR,2011,42.089
784,HRV,2011,74.132
785,IDN,2018,33.281
786,ROU,2017,45.430


In [22]:
#merging the datasets based on the year
# STEP 1: Convert each pivot dataset to LONG format
dfs_long = []

for name, df in pivot_datasets.items():
    
    temp = df.reset_index().melt(
        id_vars="year",
        var_name="country",
        value_name="value"
    )
    
    temp["indicator"] = name  # keep track of dataset source
    
    dfs_long.append(temp)


# STEP 2: Concatenate everything into one dataset
df_all = pd.concat(dfs_long, ignore_index=True)


# STEP 3: Pivot to final clean panel format
final_df = df_all.pivot_table(
    index=["year", "country"],
    columns="indicator",
    values="value"
).reset_index()




#Fill intelligently, NaN does NOT mean bad data, it means “data not available for that country-year-indicator”
final_clean = final_df.copy()

numeric_cols = final_clean.select_dtypes(include="number").columns

# 1. interpolate (time structure)
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.interpolate(limit_direction="both")
)

# 2. country mean
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.mean())
)




final_clean

indicator,year,country,Eurostat_Energy_Import_Dependency_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_Public_Debt_GDP_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025
0,2000,ALB,45.781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,ARG,NaN,-0.788999,34.277230,NaN,NaN,0.392499,16.711861,NaN,7.561
2,2000,AUS,NaN,3.359670,4.457435,52.464076,12.765,1.469756,27.475746,38.0,6.288
3,2000,AUT,65.543,3.189522,2.344868,47.091146,70.344,1.896676,24.783438,25.8,3.502
4,2000,BEL,78.164,3.716706,2.544518,53.979823,94.929,1.936195,27.084993,31.8,7.009
...,...,...,...,...,...,...,...,...,...,...,...
1897,2025,SWE,26.504,1.543167,0.679869,332.353013,46.363,3.559558,51.826313,42.4,8.398
1898,2025,TUR,68.110,3.604625,34.881160,91.185172,28.557,1.461902,26.944220,NaN,8.712
1899,2025,USA,NaN,2.106338,2.949525,90.335094,123.186,3.444858,50.705994,NaN,4.022
1900,2025,USMCA,NaN,1.931042,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
missing_by_country = final_clean.groupby("country").apply(
    lambda x: x.isnull().sum()
).drop(columns=["year"]).reset_index()
missing_by_country

indicator,country,Eurostat_Energy_Import_Dependency_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_Public_Debt_GDP_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025
0,ALB,0,25,25,25,25,25,25,25,25
1,ARG,26,0,0,26,26,0,0,26,0
2,AUS,26,0,0,0,0,0,0,0,0
3,AUT,0,0,0,0,0,0,0,0,0
4,BEL,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
72,TWN,25,25,25,25,25,0,25,25,25
73,UKR,0,21,21,21,21,21,21,21,21
74,USA,26,0,0,0,0,0,0,26,0
75,USMCA,26,0,26,26,26,26,26,26,26


In [24]:
missing_by_country.to_csv("missing_by_country_4.csv", index=False)

In [25]:
!pip install pycountry


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
#Create conversion function
import pycountry

def country_to_iso3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except:
        return None



In [27]:
df7 = pd.read_csv("TrustGov_2000_2020.csv")
df7.head()

,country,year,Approval_Not_Smoothed
0,Argentina,1983,67.252609
1,Argentina,1984,61.002979
2,Argentina,1985,58.382301
3,Argentina,1986,54.313831
4,Argentina,1987,48.490929


In [28]:
df7["country"] = df7["country"].apply(country_to_iso3)

In [139]:
df7

,country,year,Approval_Not_Smoothed
0,ARG,1983,67.252609
1,ARG,1984,61.002979
2,ARG,1985,58.382301
3,ARG,1986,54.313831
4,ARG,1987,48.490929
...,...,...,...
2215,VEN,2015,33.350620
2216,VEN,2016,29.990950
2217,VEN,2017,31.979071
2218,VEN,2018,30.125401


In [29]:
df7[df7["country"].isnull()]["country"].unique()

<StringArray>
[nan]
Length: 1, dtype: str

In [30]:
#some rows have no country name by the way
df7[df7["country"].isnull()]

,country,year,Approval_Not_Smoothed
1257,NaN,2007,52.485630
1258,NaN,2008,57.206451
1259,NaN,2009,48.144211
1260,NaN,2010,39.845322
1261,NaN,2011,34.870399
...,...,...,...
1941,NaN,2014,-0.521792
1942,NaN,2015,-1.468777
1943,NaN,2016,0.895465
1944,NaN,2017,0.838192


In [31]:
#Clening workflow
df7 = df7.dropna(subset=["country"])
df7["country_code"] = df7["country"].apply(country_to_iso3)

In [32]:
df7 = df7.rename(columns={"Approval_Not_Smoothed": "gov_trust"})

In [33]:
#merging
merged = pd.merge(
    final_clean,
    df7,
    on=["country", "year"],
    how="outer"
)

In [15]:
merged

,year,country,Eurostat_Energy_Import_Dependency_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_Public_Debt_GDP_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025,gov_trust,country_code
0,2000,ALB,45.781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2001,ALB,52.210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,ALB,53.360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,ALB,50.757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2004,ALB,47.764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3345,2021,ZAF,NaN,4.858649,4.618745,NaN,80.246,0.615218,14.876919,NaN,33.998,NaN,NaN
3346,2022,ZAF,NaN,2.058137,7.039873,NaN,74.543,0.613773,13.914134,NaN,33.265,NaN,NaN
3347,2023,ZAF,NaN,0.806105,6.075244,NaN,74.543,0.616860,11.449790,NaN,32.086,NaN,NaN
3348,2024,ZAF,NaN,0.534844,4.361153,NaN,74.543,0.616860,8.985447,NaN,32.274,NaN,NaN


In [34]:
merged = merged.drop("country_code", axis=1)

In [35]:
merged

,year,country,Eurostat_Energy_Import_Dependency_2000_2025,OECD_GDP_Growth_2000_2025,OECD_Inflation_CPI_2000_2025,OECD_Productivity_Variation_2000_2025,OECD_Public_Debt_GDP_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,OECD_Unemployment_Rate_2000_2025,gov_trust
0,2000,ALB,45.781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2001,ALB,52.210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,ALB,53.360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,ALB,50.757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2004,ALB,47.764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3345,2021,ZAF,NaN,4.858649,4.618745,NaN,80.246,0.615218,14.876919,NaN,33.998,NaN
3346,2022,ZAF,NaN,2.058137,7.039873,NaN,74.543,0.613773,13.914134,NaN,33.265,NaN
3347,2023,ZAF,NaN,0.806105,6.075244,NaN,74.543,0.616860,11.449790,NaN,32.086,NaN
3348,2024,ZAF,NaN,0.534844,4.361153,NaN,74.543,0.616860,8.985447,NaN,32.274,NaN


In [36]:
!pip install openpyxl


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
dfx = pd.read_excel("Worldwide Governance Indicators.xlsx")

In [38]:
dfx.to_csv("your_file.csv", index=False)

In [172]:
dfx

,ID variable (economy code/ gov. dimension/ year),Economy (name),Economy (code),Region,Income classification,Year,Governance dimension,Number of sources,Governance estimate (approx. -2.5 to +2.5),Standard error (estimate),...,OBI mean,PIA mean,PRS mean,RSF mean,VAB mean,VDM mean,WBS mean,WCY mean,WJP mean,WMO mean
0,ADOva1996,Andorra,ADO,NaN,NaN,1996,va,3,1.541954,0.301021,...,..,..,..,..,..,..,..,..,..,1
1,AFGva1996,Afghanistan,AFG,South Asia,Low income,1996,va,4,-2.235444,0.245080,...,..,..,..,..,..,0.246567,..,..,..,0.0625
2,AGOva1996,Angola,AGO,Sub-Saharan Africa,Lower middle income,1996,va,6,-1.746207,0.193985,...,..,..,0.25,..,..,0.320267,..,..,..,0.125
3,ALBva1996,Albania,ALB,Europe & Central Asia,Upper middle income,1996,va,5,-0.826077,0.221397,...,..,..,0.5,..,..,0.448356,..,..,..,0.25
4,AREva1996,United Arab Emirates,ARE,Middle East & North Africa,High income,1996,va,6,-0.848031,0.193985,...,..,..,0.583333,..,..,0.332567,..,..,..,0.6875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,XKXva2024,Kosovo,XKX,Europe & Central Asia,Upper middle income,2024,va,9,0.159588,0.156446,...,..,0.5,..,0.5273,..,0.607289,..,..,0.578919,..
5440,YEMva2024,"Yemen, Rep.",YEM,Middle East & North Africa,Low income,2024,va,9,-1.697182,0.178875,...,0,0.1,0.416667,0.3145,..,0.3571,..,..,..,..
5441,ZAFva2024,South Africa,ZAF,Sub-Saharan Africa,Upper middle income,2024,va,13,0.607280,0.147152,...,0.82713,..,0.833333,0.7571,..,0.650411,..,0.314754,0.619057,..
5442,ZMBva2024,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,2024,va,14,-0.407569,0.142237,...,0.339358,0.3,0.791667,0.5733,..,0.5176,..,..,0.421383,..


In [39]:
dfx["Economy (name)"] = dfx["Economy (name)"].apply(country_to_iso3)

In [21]:
dfx

,ID variable (economy code/ gov. dimension/ year),Economy (name),Economy (code),Region,Income classification,Year,Governance dimension,Number of sources,Governance estimate (approx. -2.5 to +2.5),Standard error (estimate),...,OBI mean,PIA mean,PRS mean,RSF mean,VAB mean,VDM mean,WBS mean,WCY mean,WJP mean,WMO mean
0,ADOva1996,AND,ADO,NaN,NaN,1996,va,3,1.541954,0.301021,...,..,..,..,..,..,..,..,..,..,1
1,AFGva1996,AFG,AFG,South Asia,Low income,1996,va,4,-2.235444,0.245080,...,..,..,..,..,..,0.246567,..,..,..,0.0625
2,AGOva1996,AGO,AGO,Sub-Saharan Africa,Lower middle income,1996,va,6,-1.746207,0.193985,...,..,..,0.25,..,..,0.320267,..,..,..,0.125
3,ALBva1996,ALB,ALB,Europe & Central Asia,Upper middle income,1996,va,5,-0.826077,0.221397,...,..,..,0.5,..,..,0.448356,..,..,..,0.25
4,AREva1996,ARE,ARE,Middle East & North Africa,High income,1996,va,6,-0.848031,0.193985,...,..,..,0.583333,..,..,0.332567,..,..,..,0.6875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,XKXva2024,NaN,XKX,Europe & Central Asia,Upper middle income,2024,va,9,0.159588,0.156446,...,..,0.5,..,0.5273,..,0.607289,..,..,0.578919,..
5440,YEMva2024,NaN,YEM,Middle East & North Africa,Low income,2024,va,9,-1.697182,0.178875,...,0,0.1,0.416667,0.3145,..,0.3571,..,..,..,..
5441,ZAFva2024,ZAF,ZAF,Sub-Saharan Africa,Upper middle income,2024,va,13,0.607280,0.147152,...,0.82713,..,0.833333,0.7571,..,0.650411,..,0.314754,0.619057,..
5442,ZMBva2024,ZMB,ZMB,Sub-Saharan Africa,Lower middle income,2024,va,14,-0.407569,0.142237,...,0.339358,0.3,0.791667,0.5733,..,0.5176,..,..,0.421383,..


In [40]:
dfx.columns

Index(['ID variable (economy code/ gov. dimension/ year)', 'Economy (name)',
       'Economy (code)', 'Region', 'Income classification', 'Year',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)', 'Governance score (0-100)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WM

In [41]:
dfx = dfx.drop(['ID variable (economy code/ gov. dimension/ year)',
       'Economy (code)', 'Region', 'Income classification',
       'Governance dimension', 'Number of sources',
       'Governance estimate (approx. -2.5 to +2.5)',
       'Standard error (estimate)',
       'Lower threshold (90% conf. int. estimate)',
       'Upper threshold (90% conf. int. estimate)',
       'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)',
       'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean',
       'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean',
       'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean',
       'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean',
       'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean',
       'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean',
       'WCY mean', 'WJP mean', 'WMO mean'], axis=1)

In [42]:
dfx

,Economy (name),Year,Governance score (0-100)
0,AND,1996,83.837279
1,AFG,1996,17.723405
2,AGO,1996,26.286262
3,ALB,1996,42.390843
4,ARE,1996,42.006593
...,...,...,...
5439,NaN,2024,59.642428
5440,NaN,2024,27.144335
5441,ZAF,2024,67.478144
5442,ZMB,2024,49.715769


In [43]:
dfx.rename(columns={"Economy (name)": "country"}, inplace=True)
dfx.rename(columns={"Year": "year"}, inplace=True)

In [44]:
#merging
final_merged = pd.merge(
    merged,
    dfx,
    on=["country", "year"],
    how="inner"
)

In [37]:
final_merged

,year,country,Eurostat_Energy_Import_Dependency_2000_2025,OECD_GDP_Growth_2000_2025,Inflation_Rate,Productivity_Variation,OECD_Public_Debt_GDP_2000_2025,OECD_RD_GDP_2000_2025,OECD_Tertiary_Education_2000_2025,OECD_Trust_Institutions_2000_2025,Unemployment_Variation,gov_trust,country_code,Gov_score
0,2000,ALB,45.781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.661534
1,2002,ALB,53.360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.080731
2,2003,ALB,50.757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.631023
3,2004,ALB,47.764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.665036
4,2005,ALB,49.710,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58.610539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1949,2020,ZAF,NaN,-6.168918,3.232388,NaN,80.663,0.602962,15.839705,NaN,29.210,NaN,NaN,65.736524
1950,2021,ZAF,NaN,4.858649,4.618745,NaN,80.246,0.615218,14.876919,NaN,33.998,NaN,NaN,68.059806
1951,2022,ZAF,NaN,2.058137,7.039873,NaN,74.543,0.613773,13.914134,NaN,33.265,NaN,NaN,66.731523
1952,2023,ZAF,NaN,0.806105,6.075244,NaN,74.543,0.616860,11.449790,NaN,32.086,NaN,NaN,66.688605


In [45]:
final_merged = final_merged.rename(columns={"OECD_GDP_Growth_2000_2025": "R&D expenditure (%GDP)"})
final_merged = final_merged.rename(columns={"OECD_Trust_Institutions_2000_2025": "trust_in_institutions"})
final_merged = final_merged.rename(columns={"OECD_Public_Debt_GDP_2000_2025": "Public_Debt (%GDP)"})
final_merged = final_merged.rename(columns={"Eurostat_Energy_Import_Dependency_2000_2025": "Energy_Import_Dependency"})
final_merged = final_merged.rename(columns={"OECD_Tertiary_Education_2000_2025": "Tertiary_education_attainment"})

In [46]:
final_df1 = final_merged[
    [
        "year",
        "country",
        "R&D expenditure (%GDP)",
        "trust_in_institutions",
        "Public_Debt (%GDP)",
        "Energy_Import_Dependency",
        "Tertiary_education_attainment"
     
    ]
]

In [47]:
final_merged = final_merged.rename(columns={"OECD_Inflation_CPI_2000_2025": "Inflation_Rate"})
final_merged = final_merged.rename(columns={"OECD_Unemployment_Rate_2000_2025": "Unemployment_Variation"})
final_merged = final_merged.rename(columns={"Governance score (0-100)": "Gov_score"})
final_merged = final_merged.rename(columns={"OECD_Productivity_Variation_2000_2025": "Productivity_Variation"})
final_merged = final_merged.rename(columns={"R&D expenditure (%GDP)": "GDP_Growth"})
final_df2 = final_merged[
    [
        "year",
        "country",
        "Inflation_Rate",
        "Unemployment_Variation",
        "Gov_score",
        "Productivity_Variation",
        "GDP_Growth"
        
    ]
]

In [48]:
final_df1

,year,country,R&D expenditure (%GDP),trust_in_institutions,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment
0,2000,ALB,NaN,NaN,NaN,45.781,NaN
1,2002,ALB,NaN,NaN,NaN,53.360,NaN
2,2003,ALB,NaN,NaN,NaN,50.757,NaN
3,2004,ALB,NaN,NaN,NaN,47.764,NaN
4,2005,ALB,NaN,NaN,NaN,49.710,NaN
...,...,...,...,...,...,...,...
1949,2020,ZAF,-6.168918,NaN,80.663,NaN,15.839705
1950,2021,ZAF,4.858649,NaN,80.246,NaN,14.876919
1951,2022,ZAF,2.058137,NaN,74.543,NaN,13.914134
1952,2023,ZAF,0.806105,NaN,74.543,NaN,11.449790


In [49]:
final_df2

,year,country,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth
0,2000,ALB,NaN,NaN,47.661534,NaN,NaN
1,2002,ALB,NaN,NaN,53.080731,NaN,NaN
2,2003,ALB,NaN,NaN,54.631023,NaN,NaN
3,2004,ALB,NaN,NaN,56.665036,NaN,NaN
4,2005,ALB,NaN,NaN,58.610539,NaN,NaN
...,...,...,...,...,...,...,...
1949,2020,ZAF,3.232388,29.210,65.736524,NaN,-6.168918
1950,2021,ZAF,4.618745,33.998,68.059806,NaN,4.858649
1951,2022,ZAF,7.039873,33.265,66.731523,NaN,2.058137
1952,2023,ZAF,6.075244,32.086,66.688605,NaN,0.806105


In [53]:
final_df3 = final_merged[
    [
        "year",
        "country",
        "Public_Debt (%GDP)",
        "Energy_Import_Dependency",
        "Tertiary_education_attainment",
        "Inflation_Rate",
        "Unemployment_Variation",
        "Gov_score",
        "Productivity_Variation",
        "GDP_Growth"
        
    ]
]

In [54]:
final_df3

,year,country,Public_Debt (%GDP),Energy_Import_Dependency,Tertiary_education_attainment,Inflation_Rate,Unemployment_Variation,Gov_score,Productivity_Variation,GDP_Growth
0,2000,ALB,NaN,45.781,NaN,NaN,NaN,47.661534,NaN,NaN
1,2002,ALB,NaN,53.360,NaN,NaN,NaN,53.080731,NaN,NaN
2,2003,ALB,NaN,50.757,NaN,NaN,NaN,54.631023,NaN,NaN
3,2004,ALB,NaN,47.764,NaN,NaN,NaN,56.665036,NaN,NaN
4,2005,ALB,NaN,49.710,NaN,NaN,NaN,58.610539,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1949,2020,ZAF,80.663,NaN,15.839705,3.232388,29.210,65.736524,NaN,-6.168918
1950,2021,ZAF,80.246,NaN,14.876919,4.618745,33.998,68.059806,NaN,4.858649
1951,2022,ZAF,74.543,NaN,13.914134,7.039873,33.265,66.731523,NaN,2.058137
1952,2023,ZAF,74.543,NaN,11.449790,6.075244,32.086,66.688605,NaN,0.806105


In [55]:
final_df3.to_csv("completedataset.csv", index=False)

In [314]:
#to find where saved
import os
os.getcwd()

'C:\\Users\\21626\\Desktop\\ds_lab'

In [51]:
import numpy as np

# function to calculate recovery time
def calculate_recovery_time(df, shock_year, gdp_col):

    recovery_times = []

    for country in df["country"].unique():

        country_df = df[df["country"] == country].sort_values("year")

        # get GDP value before shock
        try:
            baseline = country_df.loc[
                country_df["year"] == shock_year,
                gdp_col
            ].values[0]
        except:
            continue

        recovery_year = np.nan

        # search future years
        future_data = country_df[country_df["year"] > shock_year]

        for _, row in future_data.iterrows():

            if row[gdp_col] >= baseline:
                recovery_year = row["year"] - shock_year
                break

        # assign result to all rows of that country
        country_df = country_df.copy()
        country_df[f"recovery_{shock_year}"] = recovery_year

        recovery_times.append(country_df)

    return pd.concat(recovery_times)


# =========================
# APPLY FOR 2007
# =========================
final_df2 = calculate_recovery_time(
    final_df2,
    shock_year=2007,
    gdp_col="GDP_Growth"
)

# =========================
# APPLY FOR 2019
# =========================
final_df2 = calculate_recovery_time(
    final_df2,
    shock_year=2019,
    gdp_col="GDP_Growth"
)

In [10]:
final_df2.head(100)

NameError: name 'final_df2' is not defined

In [56]:
final_df1.to_csv("finalfinalfinal_dataset_one.csv", index=False)

final_df2.to_csv("finalfinalfinal_dataset_two.csv", index=False)